In [3]:
!pip install langchain
!pip install langchain-openai
!pip install langchain-community
!pip install langchain-chroma

In [5]:
import sqlite3
import pandas as pd

In [6]:
# Read the code below and write your observation in the next cell

conn = sqlite3.connect('eng_subtitles_database.db')
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print(cursor.fetchall())

[('zipfiles',)]


In [7]:
cursor.execute("PRAGMA table_info('zipfiles')")
cols = cursor.fetchall()
for col in cols:
    print(col[1])

num
name
content


In [8]:
df = pd.read_sql_query("""SELECT * FROM zipfiles""", conn)
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82498 entries, 0 to 82497
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   num      82498 non-null  int64 
 1   name     82498 non-null  object
 2   content  82498 non-null  object
dtypes: int64(1), object(2)
memory usage: 1.9+ MB


In [9]:
b_data = df.iloc[0, 2]

In [10]:
print(b_data[:500])  # Print the first 500 characters of the binary data

b"PK\x03\x04\x14\x00\x00\x00\x08\x00\x1c\xa9\x99V\x9fx\x96\xf0\x8c\x9e\x00\x00\x86\x9b\x01\x00;\x00\x00\x00The.Message.1976.REMASTERED.1080p.BluRay.x264-PiGNUS.EN.srt\xad\xbdm\x93\xdc\xc6\x91.\xfa\x9d\x11\xfc\x0f-}\xe1=\x11-\x9d\x06P\x85\x17\x9d\x8d\xd5%%[\xa4-Y>&u\x15>\xdf\xd0\xd3\x98\x19x\xfae\x0cts<\xfe\xf57\x9f'\xb3\n\xd9\xa4\xbc\xbb\xf7\xc6Fl\xacELW\xa2\xaa\x90\x95\x95\xafO\x16/_l6\xdf\xe0\xff\xea\xf5f\xb3Y}\xf5\xd5\xbf\xaf\xf4AQ\xae7Mx\xf9\xe2\xd7\xfe|s\xbf\xea\x8f\xcf\xab\x8f\xe3n8\xadN\xc7\xfdx\x1cVO\xe3\xf9~\xf5\xf3\xe3p\xfc\xea\xfd/o>\xbc\xfb\xf0\xe3\xef\xde\xbf|\xf1\xfbi\x18Vo\xa6\xd3\xd3<L\xab\xe1\x1f\xe7\xe18\x8f\xa7\xe37\xab\xd3\xbc\xdb~-\xc3\x1e\xfe\xa7<|\xf9\xe2\xe5\x8bR_[~S\xd6\xeb\xa2k\xf3k\xe5A\xb7\xeeb\xf5\xf2\xc5\xbb\xe3\xea|?\xac\x8e\xfdaX\x9dnW?\x9cvk>8\x9c\xe6\xf3\xean\xeao\xc6\xd3ev\x8f~\x1a\xa6\x9b\xf1\xf6\xb2\xff\x1a\xe4\xabD\xbe*d\x11\xa5#_U\xeb\xaa\xd9`\xa6\xa7\xc3\xea\xa7\xcb}\x7f8\xf4F\xf9\xa7a\x9e\x87\xe3\x9d\xcc\\\xdf\x07B!\x13\xaa\xd61n<!\xd9\xaf\xd0\x

In [10]:
import zipfile
import io

count = 0

def decode_method(binary_data):
    global count
    # Decompress the binary data using the zipfile module
    # print(count, end=" ")
    count += 1
    with io.BytesIO(binary_data) as f:
        with zipfile.ZipFile(f, 'r') as zip_file:
            # Assuming there's only one file in the ZIP archive
            subtitle_content = zip_file.read(zip_file.namelist()[0])
    
    # Now 'subtitle_content' should contain the extracted subtitle content
    return subtitle_content.decode('latin-1')  # Assuming the content is UTF-8 encoded text

In [11]:
df['file_content'] = df['content'].apply(decode_method)

df.head()

,num,name,content,file_content
0,9180533,the.message.(1976).eng.1cd,b'PK\x03\x04\x14\x00\x00\x00\x08\x00\x1c\xa9\x...,"1\r\n00:00:06,000 --> 00:00:12,074\r\nWatch an..."
1,9180583,here.comes.the.grump.s01.e09.joltin.jack.in.bo...,b'PK\x03\x04\x14\x00\x00\x00\x08\x00\x17\xb9\x...,"1\r\n00:00:29,359 --> 00:00:32,048\r\nAh! Ther..."
2,9180592,yumis.cells.s02.e13.episode.2.13.(2022).eng.1cd,b'PK\x03\x04\x14\x00\x00\x00\x08\x00L\xb9\x99V...,"1\r\n00:00:53,200 --> 00:00:56,030\r\n<i>Yumi'..."
3,9180594,yumis.cells.s02.e14.episode.2.14.(2022).eng.1cd,b'PK\x03\x04\x14\x00\x00\x00\x08\x00U\xa9\x99V...,"1\r\n00:00:06,000 --> 00:00:12,074\r\nWatch an..."
4,9180600,broker.(2022).eng.1cd,b'PK\x03\x04\x14\x00\x00\x00\x08\x001\xa9\x99V...,"ï»¿1\r\n00:00:06,000 --> 00:00:12,074\r\nWatch..."


In [12]:
!pip install langchain-chroma

In [ ]:
# Enter OpenAI API key
from getpass import getpass
OPENAI_API_KEY = getpass("Enter your OpenAI API Key here: ")

In [ ]:
# Setup Environment Variables
import os
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

In [ ]:
# OpenAI Embedding Model
from langchain_openai import OpenAIEmbeddings
openai_embed_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [16]:
# OpenAI Generating Model
from langchain_openai import ChatOpenAI
chatgpt = ChatOpenAI(model_name="gpt-4o-mini")

In [17]:

# Sample 05% of the data randomly
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Sample 30% of the data
sample_size = int(len(df) * 0.05)
sample_indices = np.random.choice(df.index, size=sample_size, replace=False)
df_sample = df.loc[sample_indices].reset_index(drop=True)

print(f"Total data: {len(df)}")
print(f"Sampled data (05%): {len(df_sample)}")
print(f"\nSample data:")
print(df_sample.head())

Total data: 82498
Sampled data (05%): 4124

Sample data:
       num                                               name  \
0  9251120                     maybe.this.time.(2014).eng.1cd   
1  9211589  down.the.shore.s01.e10.and.justice.for.all.(19...   
2  9380845  uncontrollably.fond.s01.e07.heartache.(2016).e...   
3  9301436  screen.two.s13.e04.the.precious.blood.(1996).e...   
4  9408707                          battlebots.(2015).eng.1cd   

                                             content  \
0  b'PK\x03\x04\x14\x00\x00\x00\x08\x00\x89\x9a\x...   
1  b'PK\x03\x04\x14\x00\x00\x00\x08\x007\x8f\x99V...   
2  b'PK\x03\x04\x14\x00\x00\x00\x08\x00\x8f\x19\x...   
3  b'PK\x03\x04\x14\x00\x00\x00\x08\x00[\xaa\x99V...   
4  b'PK\x03\x04\x14\x00\x00\x00\x08\x00\xf4<\x9aV...   

                                        file_content  
0  ï»¿1\r\n00:00:06,000 --> 00:00:12,074\r\nWatch...  
1  1\r\n00:00:09,275 --> 00:00:11,876\r\n¶ Oh, I ...  
2  1\r\n00:00:07,140 --> 00:00:14,220\r\n<i>Timin.

In [18]:
# Data Cleaning: Remove timestamps and unnecessary content
import re

def clean_subtitle_text(text):
    """
    Clean subtitle text by removing:
    1. Timestamps (SRT format: HH:MM:SS,mmm --> HH:MM:SS,mmm)
    2. Timestamps (VTT format: HH:MM:SS.mmm --> HH:MM:SS.mmm)
    3. Sequence numbers (1, 2, 3, etc. at the start of lines)
    4. Extra whitespace and empty lines
    5. HTML tags (if any)
    6. Speaker labels (Name:)
    """
    
    # Remove SRT format timestamps (HH:MM:SS,mmm --> HH:MM:SS,mmm)
    text = re.sub(r'\d{2}:\d{2}:\d{2},\d{3}\s*-->\s*\d{2}:\d{2}:\d{2},\d{3}', '', text)
    
    # Remove VTT format timestamps (HH:MM:SS.mmm --> HH:MM:SS.mmm)
    text = re.sub(r'\d{2}:\d{2}:\d{2}\.\d{3}\s*-->\s*\d{2}:\d{2}:\d{2}\.\d{3}', '', text)
    
    # Remove sequence numbers at the start of lines (e.g., "1", "2", "3")
    text = re.sub(r'^\d+\s*$', '', text, flags=re.MULTILINE)
    
    # Remove HTML tags (e.g., <i>, <b>, </i>, </b>)
    text = re.sub(r'<[^>]+>', '', text)
    
    # Remove speaker labels followed by colon (e.g., "JOHN:", "MARY:")
    text = re.sub(r'^[A-Z\s]+:\s*', '', text, flags=re.MULTILINE)
    
    # Remove extra whitespace (multiple spaces/tabs)
    text = re.sub(r'\s+', ' ', text)
    
    # Remove extra newlines
    text = re.sub(r'\n+', '\n', text)
    
    # Strip leading and trailing whitespace
    text = text.strip()
    
    return text

# Apply cleaning to the sampled data
df_sample['file_content_cleaned'] = df_sample['file_content'].apply(clean_subtitle_text)

print("Data Cleaning Complete!")
print(f"\nBefore Cleaning (first 300 chars):")
print(df_sample['file_content'].iloc[0][:300])
print(f"\nAfter Cleaning (first 300 chars):")
print(df_sample['file_content_cleaned'].iloc[0][:300])
print(f"\nCleaning statistics:")
print(f"Original total characters: {df_sample['file_content'].str.len().sum()}")
print(f"Cleaned total characters: {df_sample['file_content_cleaned'].str.len().sum()}")
print(f"Characters removed: {df_sample['file_content'].str.len().sum() - df_sample['file_content_cleaned'].str.len().sum()}")

Data Cleaning Complete!

Before Cleaning (first 300 chars):
ï»¿1
00:00:06,000 --> 00:00:12,074
Watch any video online with Open-SUBTITLES
Free Browser extension: osdb.link/ext

2
00:00:37,328 --> 00:00:39,706
<i>It could've been
just another summer.</i>

3
00:00:40,790 --> 00:00:43,042
<i>But as I set foot on the sand,</i>

4
00:00:43,209 --> 

After Cleaning (first 300 chars):
ï»¿1 Watch any video online with Open-SUBTITLES Free Browser extension: osdb.link/ext It could've been just another summer. But as I set foot on the sand, that summer suddenly felt different. Like it was going to be the summer that would change my life. The summer of freedom. The summer of endless p

Cleaning statistics:
Original total characters: 248841333
Cleaned total characters: 127823011
Characters removed: 121018322


In [19]:
# Document Chunker with overlapping windows
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter with chunk size and overlap
chunk_size = 500  # Token window size (approximate)
chunk_overlap = 100  # Overlap to maintain context between chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

# Create chunks for all sampled documents
documents = []

for idx, row in df_sample.iterrows():
    subtitle_text = row['file_content_cleaned']  # Use cleaned content
    subtitle_name = row['name']
    subtitle_id = row['num']
    
    # Split the document into chunks
    chunks = text_splitter.split_text(subtitle_text)
    
    # Add each chunk as a document with metadata
    for chunk_idx, chunk in enumerate(chunks):
        documents.append({
            'content': chunk,
            'metadata': {
                'subtitle_id': subtitle_id,
                'subtitle_name': subtitle_name,
                'chunk_index': chunk_idx,
                'total_chunks': len(chunks)
            }
        })

print(f"Total chunks created: {len(documents)}")
print(f"Sample chunk:")
print(f"Content: {documents[0]['content'][:200]}...")
print(f"Metadata: {documents[0]['metadata']}")

Total chunks created: 318578
Sample chunk:
Content: ï»¿1 Watch any video online with Open-SUBTITLES Free Browser extension: osdb.link/ext It could've been just another summer. But as I set foot on the sand, that summer suddenly felt different. Like it ...
Metadata: {'subtitle_id': 9251120, 'subtitle_name': 'maybe.this.time.(2014).eng.1cd', 'chunk_index': 0, 'total_chunks': 143}


In [20]:
print(documents[2])

{'content': 'call me miss. Don\'t pretend to be a gentleman when you\'re clearly not. So, what should I call you? Rude? Snob? Bitch? And you? Douche? Handsome. - Conceited. - Just like you. - Huh? Jerk! - Exactly your type! Leave me alone. Steph! Aha! Steph! I\'ll just call you Tep. Remove the "S" and the "F". By the way, I\'m Tonio. Will you still be here tomorrow? Don\'t leave yet. I\'m going to court you. I chose to walk away from you. But fate had a different plan. Councilorâ\x80¦ We\'re teaching basic English', 'metadata': {'subtitle_id': 9251120, 'subtitle_name': 'maybe.this.time.(2014).eng.1cd', 'chunk_index': 2, 'total_chunks': 143}}


In [ ]:
# Store embeddings in ChromaDB
from langchain_chroma import Chroma
from langchain_core.documents import Document

# Convert documents to LangChain Document format
langchain_docs = [
    Document(
        page_content=doc['content'],
        metadata=doc['metadata']
    )
    for doc in documents
]

# Create or connect to ChromaDB vector store
vectorstore = Chroma.from_documents(
    documents=langchain_docs,
    embedding=openai_embed_model,
    collection_name="subtitle_chunks",
    persist_directory="./chroma_db"
)

print(f"Vector store created successfully!")
print(f"Collection name: subtitle_chunks")
print(f"Number of vectors stored: {vectorstore._collection.count()}")
print(f"Persist directory: ./chroma_db")

In [ ]:
# Demonstration with Sample Text Queries
def demo_search(queries, k=5):
    """
    Demo function to test multiple search queries.
    
    Args:
        queries (List[str]): List of search queries
        k (int): Number of results per query
    """
    for query in queries:
        print(f"\n{'='*80}")
        print(f"QUERY: {query}")
        print(f"{'='*80}")
        
        # Search
        results = semantic_search(query, k=k)
        
        # Display results
        for i, result in enumerate(results, 1):
            print(f"\nMatch {i}:")
            print(f"  Similarity: {result['similarity_score']:.4f}")
            print(f"  Subtitle: {result['subtitle_name']}")
            print(f"  Content: {result['content'][:200]}...")
            print(f"  Source: Chunk {result['chunk_index']} (ID: {result['subtitle_id']})")

# Test with sample queries
test_queries = [
    "summer of freedom",
    "love and happiness",
    "fight and battle",
    "mystery and secrets"
]

print("Running Demo Search with Sample Queries...")
demo_search(test_queries, k=3)

In [ ]:
# End-to-End Pipeline: Audio to Search Results
def search_by_audio(audio_path, k=10):
    """
    Complete pipeline: Audio → Text → Embedding → Search
    
    Args:
        audio_path (str): Path to the audio file
        k (int): Number of top results to return
        
    Returns:
        dict: Complete pipeline results
    """
    print("Step 1: Converting audio to text...")
    query_text = audio_to_text(audio_path)
    
    if query_text is None:
        return {"error": "Failed to transcribe audio"}
    
    print("\nStep 2: Preprocessing query...")
    cleaned_query = preprocess_query(query_text)
    print(f"Cleaned query: {cleaned_query[:100]}...")
    
    print("\nStep 3: Searching for similar subtitles...")
    results = semantic_search(cleaned_query, k=k)
    
    return {
        "original_query": query_text,
        "cleaned_query": cleaned_query,
        "results": results,
        "total_results": len(results)
    }

# Alternative: Direct text search (without audio)
def search_by_text(query_text, k=10):
    """
    Search using text query directly.
    
    Args:
        query_text (str): The search query
        k (int): Number of top results to return
        
    Returns:
        dict: Search results
    """
    print("Preprocessing query...")
    cleaned_query = preprocess_query(query_text)
    
    print("Searching for similar subtitles...")
    results = semantic_search(cleaned_query, k=k)
    
    return {
        "original_query": query_text,
        "cleaned_query": cleaned_query,
        "results": results,
        "total_results": len(results)
    }

print("Pipeline functions created successfully!")

In [ ]:
# Semantic Search Function using ChromaDB
def semantic_search(query_text, k=5):
    """
    Search for similar subtitle chunks using semantic similarity.
    
    Args:
        query_text (str): The search query
        k (int): Number of top results to return
        
    Returns:
        List[dict]: List of results with metadata and similarity scores
    """
    # Load the vector store
    vectorstore = Chroma(
        collection_name="subtitle_chunks",
        embedding_function=openai_embed_model,
        persist_directory="./chroma_db"
    )
    
    # Search using similarity
    results = vectorstore.similarity_search_with_score(query_text, k=k)
    
    # Format results with metadata
    formatted_results = []
    for doc, score in results:
        formatted_results.append({
            'content': doc.page_content,
            'similarity_score': score,
            'metadata': doc.metadata,
            'subtitle_id': doc.metadata.get('subtitle_id'),
            'subtitle_name': doc.metadata.get('subtitle_name'),
            'chunk_index': doc.metadata.get('chunk_index')
        })
    
    return formatted_results

# Test the semantic search
print("Testing Semantic Search...")
query = "the summer of freedom"
results = semantic_search(query, k=5)

print(f"\n{'='*80}")
print(f"Query: '{query}'")
print(f"{'='*80}")
for i, result in enumerate(results, 1):
    print(f"\nResult {i}:")
    print(f"  Similarity Score: {result['similarity_score']:.4f}")
    print(f"  Subtitle: {result['subtitle_name']}")
    print(f"  Content: {result['content'][:150]}...")
    print(f"  Chunk {result['chunk_index']} of {result['metadata']['total_chunks']}")

In [ ]:
# Query Preprocessing and Embedding
from typing import List, Tuple
from sklearn.metrics.pairwise import cosine_similarity

def preprocess_query(text):
    """
    Preprocess query text (same cleaning as subtitles).
    
    Args:
        text (str): Raw query text
        
    Returns:
        str: Cleaned query text
    """
    # Use the same cleaning function
    return clean_subtitle_text(text)

def create_query_embedding(query_text):
    """
    Create embedding for the query text.
    
    Args:
        query_text (str): The search query
        
    Returns:
        np.array: Embedding vector for the query
    """
    # Clean the query
    cleaned_query = preprocess_query(query_text)
    
    # Create embedding using the same model
    query_embedding = openai_embed_model.embed_query(cleaned_query)
    
    return cleaned_query, np.array(query_embedding)

# Test with a sample query
sample_query = "the summer of freedom and endless possibilities"
cleaned, embedding = create_query_embedding(sample_query)
print(f"Original query: {sample_query}")
print(f"Cleaned query: {cleaned}")
print(f"Embedding shape: {embedding.shape}")

In [ ]:
# Audio to Text Conversion using Whisper
import librosa
import numpy as np

def audio_to_text(audio_path):
    """
    Convert audio file to text using OpenAI's Whisper model.
    
    Args:
        audio_path (str): Path to the audio file (supports mp3, wav, m4a, etc.)
    
    Returns:
        str: Transcribed text from the audio
    """
    try:
        import whisper
        
        # Load the Whisper model
        model = whisper.load_model("base")
        
        # Transcribe the audio
        result = model.transcribe(audio_path, language="en")
        transcribed_text = result['text']
        
        print(f"Transcription successful!")
        print(f"Text length: {len(transcribed_text)} characters")
        print(f"Preview: {transcribed_text[:200]}...")
        
        return transcribed_text
    except Exception as e:
        print(f"Error during transcription: {str(e)}")
        return None

# Example usage (you can replace with actual audio file)
# audio_file = "sample_audio.mp3"
# query_text = audio_to_text(audio_file)

In [ ]:
# Install audio processing libraries
!pip install openai-whisper librosa soundfile